In [1]:
import numpy as np
import pandas as pd

# Configuración de consistencia
np.random.seed(2026)
n_registros_giga = 5000

# =============================================================================
# 📦 GENERACIÓN DATASET 1: df_inventario_giga (Análisis de Stock)
# =============================================================================
categorias = ["Electrónica", "Moda", "Hogar", "Alimentos", "Deportes"]
proveedores = ["AlphaSupply", "GlobalDist", "LogiCorp", "DeltaImports", "OmegaGoods"]

df_inventario_giga = pd.DataFrame(
    {
        "SKU_Producto": [f"PROD-{i:05d}" for i in range(1, n_registros_giga + 1)],
        "Nombre_Producto": np.random.choice(
            ["Articulo_A", "Articulo_B", "Articulo_C", "Articulo_D"],
            n_registros_giga,
        ),
        "Categoria": np.random.choice(categorias, n_registros_giga),
        "Stock_Actual": np.random.randint(-10, 1200, n_registros_giga).astype(
            float
        ),  # Negativos simulan errores de sistema
        "Stock_Minimo_Deseado": np.random.randint(50, 300, n_registros_giga),
        "Costo_Unitario_USD": np.random.uniform(2.5, 850.0, n_registros_giga),
        "Proveedor": np.random.choice(proveedores, n_registros_giga),
    }
)

# Inyectar anomalías de inventario (Nulos intencionales en Stock Actual y Proveedor)
df_inventario_giga.loc[
    np.random.choice(df_inventario_giga.index, 150, replace=False), "Stock_Actual"
] = np.nan
df_inventario_giga.loc[
    np.random.choice(df_inventario_giga.index, 80, replace=False), "Proveedor"
] = "  Proveedor_Incógnito  "

# =============================================================================
# 🛡️ GENERACIÓN DATASET 2: df_transacciones_giga (Detección de Fraudes)
# =============================================================================
metodos_pago = ["Tarjeta_Credito", "PayPal", "Transferencia", "Cripto"]
dispositivos = ["Mobile_App", "Web_Desktop", "Tablet", "Smart_TV"]

df_transacciones_giga = pd.DataFrame(
    {
        "ID_Transaccion": [f"TX-{i:06d}" for i in range(1, n_registros_giga + 1)],
        "ID_Usuario": np.random.randint(100001, 105000, n_registros_giga),
        "Monto_USD": np.random.exponential(scale=150, size=n_registros_giga)
        + np.random.uniform(5, 50, n_registros_giga),
        "Metodo_Pago": np.random.choice(metodos_pago, n_registros_giga),
        "Dispositivo": np.random.choice(dispositivos, n_registros_giga),
        "Puntaje_Riesgo": np.random.uniform(0.0, 1.0, n_registros_giga),
        "Pais_Origen": np.random.choice(
            ["Colombia", "México", "España", "Argentina", "EEUU", None],
            n_registros_giga,
            p=[0.3, 0.2, 0.2, 0.1, 0.1, 0.1],
        ),
    }
)

# Forzar transacciones fraudulentas extremas (Outliers)
df_transacciones_giga.loc[
    np.random.choice(df_transacciones_giga.index, 40, replace=False), "Monto_USD"
] = np.random.uniform(15000, 45000, 40)
df_transacciones_giga.loc[
    np.random.choice(df_transacciones_giga.index, 40, replace=False),
    "Puntaje_Riesgo",
] = np.random.uniform(0.95, 1.0, 40)

# Duplicados idénticos en fraude para auditoría
tx_duplicadas = df_transacciones_giga.sample(n=65, random_state=2026)
df_transacciones_giga = pd.concat(
    [df_transacciones_giga, tx_duplicadas], ignore_index=True
)

print("✅ Entorno de simulación masiva listo para mañana.")
print(f"-> df_inventario_giga: {df_inventario_giga.shape} registros.")
print(f"-> df_transacciones_giga: {df_transacciones_giga.shape} registros.")


✅ Entorno de simulación masiva listo para mañana.
-> df_inventario_giga: (5000, 7) registros.
-> df_transacciones_giga: (5065, 7) registros.


# 📉 PRÁCTICA AVANZADA: Gestión de Inventarios y Detección de Fraude

## 🗂️ Temática A: Análisis de Inventarios / Stock (10 Ejercicios)
* **Enfoque:** Auditoría de almacén, optimización de quiebres de stock, detección de mermas financieras ocultas y limpieza de registros de proveedores.



## 🛠️ Ejercicio 1 (Nivel: Fácil) - Diagnóstico de Vacíos en Almacén
* **Contexto:** El Gerente de Logística sospecha que el sistema central falló durante la última toma de inventario físico, dejando varios productos sin registrar en la columna de existencias actuales. Necesita un reporte urgente para saber la magnitud del problema.
* **Objetivo:** Calcular la cantidad exacta de valores nulos o vacíos (`NaN`) presentes en la columna `Stock_Actual` de nuestro dataset `df_inventario_giga` y mostrar en pantalla únicamente las filas que contienen esta falla para su revisión.
* **Pista conceptual (Oculta en Spoiler - Haz clic para expandir):**
  <details>
    <summary>💡 Ver pista paso a paso</summary>
    1. Utiliza el método de detección de nulos en combinación con una suma para obtener el conteo total.<br>
    2. Usa ese mismo validador de nulos como una máscara de filtrado condicional dentro de los corchetes de tu DataFrame para aislar y mostrar los registros afectados.
  </details>


### Exploracion archivo ###

In [12]:
#Visualizar dataframe
df_inventario_giga.head()

,SKU_Producto,Nombre_Producto,Categoria,Stock_Actual,Stock_Minimo_Deseado,Costo_Unitario_USD,Proveedor
0,PROD-00001,Articulo_B,Moda,573.0,142,315.658325,GlobalDist
1,PROD-00002,Articulo_C,Deportes,112.0,121,410.316674,AlphaSupply
2,PROD-00003,Articulo_C,Electrónica,835.0,102,476.147004,AlphaSupply
3,PROD-00004,Articulo_A,Deportes,790.0,283,66.730196,OmegaGoods
4,PROD-00005,Articulo_B,Moda,620.0,96,568.783278,LogiCorp


In [13]:
#Visualizar dataframe
df_transacciones_giga.head()

,ID_Transaccion,ID_Usuario,Monto_USD,Metodo_Pago,Dispositivo,Puntaje_Riesgo,Pais_Origen
0,TX-000001,104593,54.123836,Tarjeta_Credito,Smart_TV,0.073472,España
1,TX-000002,103257,154.804227,PayPal,Smart_TV,0.474890,España
2,TX-000003,100903,178.686117,Transferencia,Web_Desktop,0.140789,Argentina
3,TX-000004,103801,122.452513,PayPal,Smart_TV,0.350563,México
4,TX-000005,104817,214.007065,Cripto,Web_Desktop,0.533955,Colombia


In [14]:
#Revisar información de las columnas del dataframe inventario
#Se evidencian faltantes de 150 registros en la columna Stock_Actual
df_inventario_giga.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   SKU_Producto          5000 non-null   str    
 1   Nombre_Producto       5000 non-null   str    
 2   Categoria             5000 non-null   str    
 3   Stock_Actual          4850 non-null   float64
 4   Stock_Minimo_Deseado  5000 non-null   int64  
 5   Costo_Unitario_USD    5000 non-null   float64
 6   Proveedor             5000 non-null   str    
dtypes: float64(2), int64(1), str(4)
memory usage: 459.3 KB


In [15]:
#Revisar información de las columnas del dataframe inventario
#Se evidencian faltantes de 499 registros en la columna Pais_Origen
df_transacciones_giga.info()

<class 'pandas.DataFrame'>
RangeIndex: 5065 entries, 0 to 5064
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   ID_Transaccion  5065 non-null   str    
 1   ID_Usuario      5065 non-null   int64  
 2   Monto_USD       5065 non-null   float64
 3   Metodo_Pago     5065 non-null   str    
 4   Dispositivo     5065 non-null   str    
 5   Puntaje_Riesgo  5065 non-null   float64
 6   Pais_Origen     4566 non-null   str    
dtypes: float64(2), int64(1), str(4)
memory usage: 447.3 KB


In [16]:
#Comprobar los datos faltantes por columna en dataframe inventario
df_inventario_giga.isna().sum()

SKU_Producto              0
Nombre_Producto           0
Categoria                 0
Stock_Actual            150
Stock_Minimo_Deseado      0
Costo_Unitario_USD        0
Proveedor                 0
dtype: int64

In [17]:
#Visualizar los registros faltantes/nulos
df_inventario_giga[df_inventario_giga['Stock_Actual'].isnull()]


,SKU_Producto,Nombre_Producto,Categoria,Stock_Actual,Stock_Minimo_Deseado,Costo_Unitario_USD,Proveedor
7,PROD-00008,Articulo_A,Deportes,NaN,61,444.482246,DeltaImports
47,PROD-00048,Articulo_A,Deportes,NaN,180,145.878338,AlphaSupply
66,PROD-00067,Articulo_C,Deportes,NaN,136,178.314448,DeltaImports
79,PROD-00080,Articulo_C,Hogar,NaN,131,592.326029,GlobalDist
132,PROD-00133,Articulo_A,Moda,NaN,180,298.241932,GlobalDist
...,...,...,...,...,...,...,...
4686,PROD-04687,Articulo_A,Hogar,NaN,216,792.960039,GlobalDist
4699,PROD-04700,Articulo_A,Electrónica,NaN,249,75.500333,DeltaImports
4770,PROD-04771,Articulo_C,Alimentos,NaN,207,614.435389,OmegaGoods
4846,PROD-04847,Articulo_B,Alimentos,NaN,99,178.868601,OmegaGoods


In [18]:
#Comprobar los datos faltantes por columna en dataframe transacciones
df_transacciones_giga.isna().sum()

ID_Transaccion      0
ID_Usuario          0
Monto_USD           0
Metodo_Pago         0
Dispositivo         0
Puntaje_Riesgo      0
Pais_Origen       499
dtype: int64

In [19]:
#Visualizar los registros faltantes/nulos
df_transacciones_giga[df_transacciones_giga['Pais_Origen'].isnull()]

,ID_Transaccion,ID_Usuario,Monto_USD,Metodo_Pago,Dispositivo,Puntaje_Riesgo,Pais_Origen
15,TX-000016,104545,29.122524,Cripto,Tablet,0.693299,NaN
22,TX-000023,101397,60.359167,Transferencia,Smart_TV,0.629854,NaN
51,TX-000052,100604,30.944687,Transferencia,Smart_TV,0.690817,NaN
81,TX-000082,101279,33.127767,Cripto,Web_Desktop,0.758757,NaN
90,TX-000091,104673,54.592212,Cripto,Tablet,0.662596,NaN
...,...,...,...,...,...,...,...
5025,TX-004685,101028,56.774121,PayPal,Smart_TV,0.114301,NaN
5032,TX-001184,100359,52.266078,Transferencia,Mobile_App,0.515058,NaN
5041,TX-004451,100636,95.342423,Tarjeta_Credito,Web_Desktop,0.907464,NaN
5047,TX-001506,100036,194.391000,Cripto,Web_Desktop,0.676862,NaN


## Solución ##

In [23]:
#Contar valores nulos/vacíos en columna 'Stock_Actual'
conteo_nulos_inv = df_inventario_giga['Stock_Actual'].isnull().sum()
print(f'Valores faltantes en inventario {conteo_nulos_inv}')

Valores faltantes en inventario 150


## 🛠️ Ejercicio 2 (Nivel: Fácil) - Limpieza de Espacios en Proveedores
* **Contexto:** Al revisar el catálogo de proveedores, el equipo de compras nota que el buscador no encuentra de forma correcta al proveedor `"Proveedor_Incógnito"`. Esto ocurre porque el sistema de registro inyectó accidentalmente espacios en blanco invisibles al inicio y al final del texto durante la carga masiva.
* **Objetivo:** Aplicar una limpieza definitiva sobre la columna `Proveedor` de `df_inventario_giga` para remover todos los espacios en blanco invisibles de los extremos.
* **Pista conceptual (Oculta en Spoiler - Haz clic para expandir):**
  <details>
    <summary>💡 Ver pista paso a paso</summary>
    1. Accede a las funciones de manejo de cadenas de texto vectorizadas en Pandas usando el prefijo `.str`.<br>
    2. Utiliza el método que recorta o remueve los espacios exteriores y reasigna el resultado de vuelta a la misma columna `Proveedor` para guardar los cambios permanentemente.
  </details>


In [ ]:
#Buscar registros con espacios en blanco en columna 'Proveedor'
df_inventario_giga[df_inventario_giga['Proveedor'].str.contains(r"\s", na=False)]

,SKU_Producto,Nombre_Producto,Categoria,Stock_Actual,Stock_Minimo_Deseado,Costo_Unitario_USD,Proveedor
15,PROD-00016,Articulo_C,Deportes,718.0,252,324.712836,Proveedor_Incógnito
41,PROD-00042,Articulo_B,Moda,224.0,68,682.362082,Proveedor_Incógnito
75,PROD-00076,Articulo_D,Hogar,429.0,163,434.137821,Proveedor_Incógnito
91,PROD-00092,Articulo_D,Alimentos,533.0,68,162.516235,Proveedor_Incógnito
188,PROD-00189,Articulo_D,Deportes,1185.0,220,546.624873,Proveedor_Incógnito
...,...,...,...,...,...,...,...
4676,PROD-04677,Articulo_A,Moda,530.0,138,382.453814,Proveedor_Incógnito
4847,PROD-04848,Articulo_B,Moda,752.0,87,232.047806,Proveedor_Incógnito
4879,PROD-04880,Articulo_C,Moda,NaN,283,462.259957,Proveedor_Incógnito
4915,PROD-04916,Articulo_A,Alimentos,671.0,68,551.994591,Proveedor_Incógnito


In [30]:
#Limpiar espacios en blanco en columna 'Proveedor'
df_inventario_giga['Proveedor'] = df_inventario_giga['Proveedor'].str.strip()

In [31]:
#Buscar los registros que correspondan a 'Proveedor_Incógnito'
df_inventario_giga[df_inventario_giga['Proveedor'] == 'Proveedor_Incógnito']

,SKU_Producto,Nombre_Producto,Categoria,Stock_Actual,Stock_Minimo_Deseado,Costo_Unitario_USD,Proveedor
15,PROD-00016,Articulo_C,Deportes,718.0,252,324.712836,Proveedor_Incógnito
41,PROD-00042,Articulo_B,Moda,224.0,68,682.362082,Proveedor_Incógnito
75,PROD-00076,Articulo_D,Hogar,429.0,163,434.137821,Proveedor_Incógnito
91,PROD-00092,Articulo_D,Alimentos,533.0,68,162.516235,Proveedor_Incógnito
188,PROD-00189,Articulo_D,Deportes,1185.0,220,546.624873,Proveedor_Incógnito
...,...,...,...,...,...,...,...
4676,PROD-04677,Articulo_A,Moda,530.0,138,382.453814,Proveedor_Incógnito
4847,PROD-04848,Articulo_B,Moda,752.0,87,232.047806,Proveedor_Incógnito
4879,PROD-04880,Articulo_C,Moda,NaN,283,462.259957,Proveedor_Incógnito
4915,PROD-04916,Articulo_A,Alimentos,671.0,68,551.994591,Proveedor_Incógnito


## 🛠️ Ejercicio 3 (Nivel: Fácil) - Remoción de Registros de Prueba (Duplicados)
* **Contexto:** El departamento de TI nos informa que durante la generación del inventario se ejecutó un script de prueba que clonó aleatoriamente varios registros de productos de manera exacta. Estos clones alteran el conteo real de existencias físicas y el valor total del inventario.
* **Objetivo:** Identificar y eliminar de forma definitiva todas las filas que sean duplicados idénticos en `df_inventario_giga`, asegurándote de conservar únicamente la primera aparición de cada registro.
* **Pista conceptual (Oculta en Spoiler - Haz clic para expandir):**
  <details>
    <summary>💡 Ver pista paso a paso</summary>
    1. Utiliza el método nativo de Pandas encargado de eliminar registros duplicados en todo el DataFrame.<br>
    2. Recuerda aplicar la reasignación de la variable original para que el cambio se guarde de forma permanente y el inventario quede depurado.
  </details>


In [36]:
#Validar registros duplicados
df_inventario_giga[df_inventario_giga.duplicated()]


,SKU_Producto,Nombre_Producto,Categoria,Stock_Actual,Stock_Minimo_Deseado,Costo_Unitario_USD,Proveedor


In [ ]:
#Eliminar registros duplicados conservando el registro que apareció primero
df_inventario_giga = df_inventario_giga.drop_duplicates(keep='first')

## 🔍 Ejercicio 4 (Nivel: Medio) - Imputación de Stock por Mediana de Categoría
* **Contexto:** El Gerente de Logística rechaza rotundamente rellenar las celdas vacías de `Stock_Actual` con un promedio global. Nos explica que los productos de "Electrónica" (como teléfonos o laptops) se manejan en volúmenes mucho menores en el almacén que los productos de "Alimentos" o "Moda". Usar una métrica general distorsionaría gravemente la realidad del inventario.
* **Objetivo:** Calcular la **mediana** de la columna `Stock_Actual` de forma separada para cada tipo de `Categoria`. Después, utilizando filtros condicionales avanzados (máscaras con el operador `&` y `.loc`), rellenar los valores nulos (`NaN`) de la columna `Stock_Actual` usando exclusivamente la mediana de la categoría a la que pertenece ese producto.
* **Pista conceptual (Oculta en Spoiler - Haz clic para expandir):**
  <details>
    <summary>💡 Ver pista paso a paso</summary>
    1. Revisa cuáles son las categorías únicas usando `df_inventario_giga['Categoria'].unique()`. Hay 5 categorías en total.<br>
    2. Para cada categoría (por ejemplo, "Electrónica"), calcula su mediana filtrando el DataFrame y aplicando `.median()`.<br>
    3. Construye una máscara doble con `.loc[(df_inventario_giga['Categoria'] == 'Electrónica') & (df_inventario_giga['Stock_Actual'].isnull()), 'Stock_Actual']` y asígnale la mediana calculada.<br>
    4. Repite este mismo proceso estructurado para las otras 4 categorías restantes.
  </details>


In [37]:
#Revisar categórias únicas
df_inventario_giga['Categoria'].unique()

<ArrowStringArray>
['Moda', 'Deportes', 'Electrónica', 'Hogar', 'Alimentos']
Length: 5, dtype: str

In [43]:
#Calcular mediana por categória
mediana_moda = df_inventario_giga[df_inventario_giga['Categoria'] == 'Moda']['Stock_Actual'].median()
mediana_deportes = df_inventario_giga[df_inventario_giga['Categoria'] == 'Deportes']['Stock_Actual'].median()
mediana_electronica = df_inventario_giga[df_inventario_giga['Categoria'] == 'Electrónica']['Stock_Actual'].median()
mediana_hogar = df_inventario_giga[df_inventario_giga['Categoria'] == 'Hogar']['Stock_Actual'].median()
mediana_alimentos = df_inventario_giga[df_inventario_giga['Categoria'] == 'Alimentos']['Stock_Actual'].median()

In [44]:
#Imputar a valores vacíos, la mediana de cada categoría
df_inventario_giga.loc[(df_inventario_giga['Categoria'] == 'Moda') & (df_inventario_giga['Stock_Actual'].isnull()), 'Stock_Actual'] = mediana_moda
df_inventario_giga.loc[(df_inventario_giga['Categoria'] == 'Deportes') & (df_inventario_giga['Stock_Actual'].isnull()), 'Stock_Actual'] = mediana_deportes
df_inventario_giga.loc[(df_inventario_giga['Categoria'] == 'Electrónica') & (df_inventario_giga['Stock_Actual'].isnull()), 'Stock_Actual'] = mediana_electronica
df_inventario_giga.loc[(df_inventario_giga['Categoria'] == 'Hogar') & (df_inventario_giga['Stock_Actual'].isnull()), 'Stock_Actual'] = mediana_hogar
df_inventario_giga.loc[(df_inventario_giga['Categoria'] == 'Alimentos') & (df_inventario_giga['Stock_Actual'].isnull()), 'Stock_Actual'] = mediana_alimentos


In [45]:
#Visualizar stocks actuales en ceros/vacíos/nulos
df_inventario_giga[df_inventario_giga['Stock_Actual'].isnull()]

,SKU_Producto,Nombre_Producto,Categoria,Stock_Actual,Stock_Minimo_Deseado,Costo_Unitario_USD,Proveedor


## 🛠️ Ejercicio 5 (Nivel: Fácil) - Creación de la Métrica de Valor del Inventario
* **Contexto:** El Director Financiero necesita conocer el capital total que la empresa tiene inmovilizado en el almacén. Para ello, requiere una nueva columna en el dataset que calcule el valor económico total por cada producto disponible en stock.
* **Objetivo:** Crear una nueva columna llamada `Valor_Inventario_USD` en `df_inventario_giga`. El valor de esta columna debe ser el resultado de multiplicar el `Stock_Actual` por el `Costo_Unitario_USD` de cada registro.
* **Pista conceptual (Oculta en Spoiler - Haz clic para expandir):**
  <details>
    <summary>💡 Ver pista paso a paso</summary>
    1. Para declarar una nueva columna en Pandas, simplemente escribe el nombre del DataFrame seguido de la nueva columna entre corchetes y comillas, por ejemplo: `df_inventario_giga['Valor_Inventario_USD']`.<br>
    2. Al otro lado del signo igual (`=`), realiza la multiplicación directa de las dos columnas numéricas existentes de forma vectorizada. Pandas se encargará de hacer la operación fila por fila automáticamente.
  </details>


In [49]:
df_inventario_giga['Valor_Inventario_USD'] = (df_inventario_giga['Stock_Actual'] * df_inventario_giga['Costo_Unitario_USD'])
df_inventario_giga.head()

,SKU_Producto,Nombre_Producto,Categoria,Stock_Actual,Stock_Minimo_Deseado,Costo_Unitario_USD,Proveedor,Valor_Inventario_USD
0,PROD-00001,Articulo_B,Moda,573.0,142,315.658325,GlobalDist,180872.220276
1,PROD-00002,Articulo_C,Deportes,112.0,121,410.316674,AlphaSupply,45955.467516
2,PROD-00003,Articulo_C,Electrónica,835.0,102,476.147004,AlphaSupply,397582.748446
3,PROD-00004,Articulo_A,Deportes,790.0,283,66.730196,OmegaGoods,52716.855152
4,PROD-00005,Articulo_B,Moda,620.0,96,568.783278,LogiCorp,352645.632351


## 🔍 Ejercicio 6 (Nivel: Medio) - Detección y Aislamiento de Quiebres de Stock Críticos
* **Contexto:** Logística está perdiendo ventas debido a retrasos en las alertas de reabastecimiento. Necesitan identificar de inmediato aquellos productos que se encuentran en una situación crítica, es decir, donde las existencias disponibles están por debajo del mínimo de seguridad establecido.
* **Objetivo:** Filtrar el dataset `df_inventario_giga` para encontrar únicamente los registros donde el `Stock_Actual` sea estrictamente menor que el `Stock_Minimo_Deseado`. Guarda este subconjunto de alerta en un nuevo DataFrame independiente llamado `df_alerta_reabastecimiento` asegurando una copia profunda, y despliega las primeras 5 filas en pantalla.
* **Pista conceptual (Oculta en Spoiler - Haz clic para expandir):**
  <details>
    <summary>💡 Ver pista paso a paso</summary>
    1. Construye una máscara lógica comparando directamente las dos columnas involucradas con el operador menor que (`<`).<br>
    2. Pasa esa condición dentro de los corchetes de tu DataFrame original para aislar las filas afectadas.<br>
    3. Finaliza la asignación aplicando el método `.copy()` para congelar los datos en el nuevo DataFrame de forma independiente y utiliza `.head()` para visualizar el extracto solicitado.
  </details>


In [50]:
df_alerta_reabastecimiento = df_inventario_giga[df_inventario_giga['Stock_Actual'] < df_inventario_giga['Stock_Minimo_Deseado']].copy()
df_alerta_reabastecimiento.head(5)

,SKU_Producto,Nombre_Producto,Categoria,Stock_Actual,Stock_Minimo_Deseado,Costo_Unitario_USD,Proveedor,Valor_Inventario_USD
1,PROD-00002,Articulo_C,Deportes,112.0,121,410.316674,AlphaSupply,45955.467516
8,PROD-00009,Articulo_A,Hogar,69.0,269,841.612081,LogiCorp,58071.233613
22,PROD-00023,Articulo_D,Hogar,222.0,231,725.911752,AlphaSupply,161152.408938
23,PROD-00024,Articulo_D,Moda,70.0,89,500.212198,OmegaGoods,35014.853893
36,PROD-00037,Articulo_A,Deportes,232.0,274,493.137721,AlphaSupply,114407.951180


## 🔍 Ejercicio 7 (Nivel: Medio) - Identificación de Errores Críticos de Sistema (Stock Negativo)
* **Contexto:** Al revisar los resultados, notas un patrón preocupante: hay productos cuyo `Stock_Actual` muestra números negativos (por ejemplo, `-5`). El Gerente de Logística te explica que esto es un error grave del software del almacén cuando procesa devoluciones de forma incorrecta. Necesita aislar estos errores para corregirlos manualmente.
* **Objetivo:** Filtrar el DataFrame original `df_inventario_giga` para extraer todas las filas donde el `Stock_Actual` sea estrictamente menor que cero (`< 0`). Guarda estos registros erróneos en un nuevo DataFrame llamado `df_errores_sistema` mediante una copia profunda y cuenta cuántas filas totales se detectaron con esta falla.
* **Pista conceptual (Oculta en Spoiler - Haz clic para expandir):**
  <details>
    <summary>💡 Ver pista paso a paso</summary>
    1. Construye una máscara lógica comparando la columna `Stock_Actual` con el valor `0` mediante el operador menor que.<br>
    2. Aplica esta máscara usando corchetes sobre el DataFrame original y añade `.copy()` al final para crear el nuevo contenedor.<br>
    3. Para obtener el conteo total de filas afectadas, mide la longitud del nuevo DataFrame con la función global `len()` o usando el atributo `.shape[0]`.
  </details>
